In [ ]:
# ============================================================
# COMPLETE SETUP
# ============================================================

from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm


# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning"
)

DATASET_DIR = (
    PROJECT_ROOT / "Dataset"
)

RAW_DIR = (
    DATASET_DIR / "Raw"
)

PROCESSED_DIR = (
    DATASET_DIR / "Processed"
)

CAPTIONS_DIR = (
    PROCESSED_DIR / "Captions"
)

IMAGES_OUTPUT_DIR = (
    PROCESSED_DIR / "Images"
)


# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------

CAPTIONS_PATH = (
    CAPTIONS_DIR
    / "captions_processed.csv"
)


print("=" * 70)
print("NOTEBOOK 4 — IMAGE PREPARATION")
print("=" * 70)

print("\nProject:")
print(PROJECT_ROOT)

print("\nCaptions input:")
print(CAPTIONS_PATH)

print("\nCaptions file exists:")
print(CAPTIONS_PATH.exists())


assert PROJECT_ROOT.exists(), (
    "Project folder not found."
)

assert CAPTIONS_PATH.exists(), (
    "captions_processed.csv not found."
)


print("\n✓ Drive mounted")
print("✓ Project found")
print("✓ Processed captions found")

Mounted at /content/drive
NOTEBOOK 4 — IMAGE PREPARATION

Project:
/content/drive/MyDrive/Image_captioning

Captions input:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv

Captions file exists:
True

✓ Drive mounted
✓ Project found
✓ Processed captions found


Load Captions and Extract Unique Images

In [ ]:
# ============================================================
# LOAD PROCESSED CAPTIONS
# ============================================================

captions_df = pd.read_csv(
    CAPTIONS_PATH
)


print("=" * 70)
print("PROCESSED CAPTIONS LOADED")
print("=" * 70)

print(
    "\nCaption records:",
    len(captions_df)
)

print(
    "Unique images:",
    captions_df["image_id"].nunique()
)

print(
    "\nDataset distribution:"
)

print(
    captions_df["dataset"]
    .value_counts()
)


# ------------------------------------------------------------
# ONE ROW PER UNIQUE IMAGE
# ------------------------------------------------------------

images_df = (
    captions_df[
        [
            "image_id",
            "dataset",
            "image_name"
        ]
    ]
    .drop_duplicates(
        subset="image_id"
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 70)
print("UNIQUE IMAGE TABLE CREATED")
print("=" * 70)

print(
    "\nShape:",
    images_df.shape
)

print("\nDataset distribution:")

print(
    images_df["dataset"]
    .value_counts()
)


assert (
    len(images_df) == 39874
)

assert (
    images_df["image_id"]
    .nunique()
    == len(images_df)
)


print("\n✓ Unique image list verified")

PROCESSED CAPTIONS LOADED

Caption records: 199369
Unique images: 39874

Dataset distribution:
dataset
Flickr30K    158914
Flickr8K      40455
Name: count, dtype: int64

UNIQUE IMAGE TABLE CREATED

Shape: (39874, 3)

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: count, dtype: int64

✓ Unique image list verified


Inspect Raw Dataset Folders

In [ ]:
# ============================================================
# INSPECT RAW DATASET STRUCTURE
# ============================================================

print("=" * 70)
print("RAW DATASET STRUCTURE")
print("=" * 70)


print("\nRAW directory:")
print(RAW_DIR)

print(
    "\nRAW directory exists:",
    RAW_DIR.exists()
)


assert RAW_DIR.exists(), (
    "Dataset/Raw directory not found."
)


print("\nTop-level contents:\n")

for item in sorted(RAW_DIR.iterdir()):

    if item.is_dir():

        print(
            "[DIR] ",
            item.name
        )

    else:

        print(
            "[FILE]",
            item.name
        )


print("\n✓ Raw directory inspected")

RAW DATASET STRUCTURE

RAW directory:
/content/drive/MyDrive/Image_captioning/Dataset/Raw

RAW directory exists: True

Top-level contents:

[DIR]  Flickr30k
[DIR]  Flickr8K

✓ Raw directory inspected


Locate JPEG Image Directories

In [ ]:
# ============================================================
# LOCATE IMAGE FILES — CORRECTED
# ============================================================

FLICKR8K_ROOT = RAW_DIR / "Flickr8K"

FLICKR30K_ROOT = (
    RAW_DIR
    / "Flickr30k"
    / "flickr30k"
)


print("=" * 70)
print("LOCATING IMAGE FILES")
print("=" * 70)


print("\nFlickr8K root:")
print(FLICKR8K_ROOT)

print(
    "Exists:",
    FLICKR8K_ROOT.exists()
)


print("\nFlickr30K IMAGE root:")
print(FLICKR30K_ROOT)

print(
    "Exists:",
    FLICKR30K_ROOT.exists()
)


assert FLICKR8K_ROOT.exists(), (
    "Flickr8K directory not found."
)

assert FLICKR30K_ROOT.exists(), (
    "Nested Flickr30K directory not found."
)


# ------------------------------------------------------------
# FIND IMAGE FILES
# ------------------------------------------------------------

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png"
}


def find_images(root_dir):

    return [
        path
        for path in root_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]


print("\nFinding Flickr8K images...")

flickr8k_image_paths = find_images(
    FLICKR8K_ROOT
)

print(
    "Flickr8K image files found:",
    len(flickr8k_image_paths)
)


print("\nFinding Flickr30K images...")

flickr30k_image_paths = find_images(
    FLICKR30K_ROOT
)

print(
    "Flickr30K image files found:",
    len(flickr30k_image_paths)
)


assert len(flickr8k_image_paths) > 0
assert len(flickr30k_image_paths) > 0


print("\nFlickr8K sample:")

for path in flickr8k_image_paths[:3]:
    print(path)


print("\nFlickr30K sample:")

for path in flickr30k_image_paths[:3]:
    print(path)


print("\n✓ IMAGE FILES LOCATED SUCCESSFULLY")

LOCATING IMAGE FILES

Flickr8K root:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr8K
Exists: True

Flickr30K IMAGE root:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr30k/flickr30k
Exists: True

Finding Flickr8K images...
Flickr8K image files found: 8091

Finding Flickr30K images...
Flickr30K image files found: 63566

Flickr8K sample:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr8K/Images/378453580_21d688748e.jpg
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr8K/Images/379006645_b9a2886b51.jpg
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr8K/Images/380034515_4fbdfa6b26.jpg

Flickr30K sample:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr30k/flickr30k/flickr30k_images/7808252542.jpg
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr30k/flickr30k/flickr30k_images/7808530252.jpg
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr30k/flickr30k/flickr30k_images/7808717546.jpg

✓ IMAGE FILES LOCATED SUCCE

5 Build Fast Filename → Path Lookup

In [ ]:
# ============================================================
# BUILD FAST IMAGE PATH LOOKUPS
# ============================================================

print("=" * 70)
print("BUILDING IMAGE LOOKUPS")
print("=" * 70)


# ------------------------------------------------------------
# FLICKR8K LOOKUP
# ------------------------------------------------------------

flickr8k_lookup = {}

for path in flickr8k_image_paths:

    filename = path.name

    if filename not in flickr8k_lookup:

        flickr8k_lookup[
            filename
        ] = str(path)


# ------------------------------------------------------------
# FLICKR30K LOOKUP
# ------------------------------------------------------------

flickr30k_lookup = {}

for path in flickr30k_image_paths:

    filename = path.name

    if filename not in flickr30k_lookup:

        flickr30k_lookup[
            filename
        ] = str(path)


print(
    "\nFlickr8K lookup entries:",
    len(flickr8k_lookup)
)

print(
    "Flickr30K lookup entries:",
    len(flickr30k_lookup)
)


print("\n✓ Fast filename lookups created")

BUILDING IMAGE LOOKUPS

Flickr8K lookup entries: 8091
Flickr30K lookup entries: 31783

✓ Fast filename lookups created


6 Match Every Image to Its Actual Path

In [ ]:
# ============================================================
# MATCH IMAGE IDS TO ACTUAL FILE PATHS
# ============================================================

print("=" * 70)
print("MATCHING IMAGES TO FILE PATHS")
print("=" * 70)


def get_image_path(row):

    dataset = row["dataset"]
    image_name = row["image_name"]

    if dataset == "Flickr8K":

        return flickr8k_lookup.get(
            image_name
        )

    elif dataset == "Flickr30K":

        return flickr30k_lookup.get(
            image_name
        )

    return None


images_df["image_path"] = (
    images_df.apply(
        get_image_path,
        axis=1
    )
)


images_df["file_found"] = (
    images_df["image_path"]
    .notna()
)


print(
    "\nTotal images:",
    len(images_df)
)

print(
    "Images found:",
    images_df["file_found"].sum()
)

print(
    "Images missing:",
    (~images_df["file_found"]).sum()
)


print("\nFound by dataset:")

print(
    images_df.groupby(
        "dataset"
    )["file_found"]
    .agg(
        [
            "count",
            "sum"
        ]
    )
)


print("\nMissing image examples:")

missing_images = images_df[
    ~images_df["file_found"]
]

if len(missing_images) > 0:

    print(
        missing_images.head(
            20
        ).to_string(
            index=False
        )
    )

else:

    print(
        "None"
    )


print("\n✓ Image matching complete")

MATCHING IMAGES TO FILE PATHS

Total images: 39874
Images found: 39874
Images missing: 0

Found by dataset:
           count    sum
dataset                
Flickr30K  31783  31783
Flickr8K    8091   8091

Missing image examples:
None

✓ Image matching complete


7 — Handle Missing Image Mappings

In [ ]:
# ============================================================
# ANALYZE MISSING IMAGE FILES
# ============================================================

print("=" * 70)
print("MISSING IMAGE ANALYSIS")
print("=" * 70)


missing_images = (
    images_df[
        ~images_df["file_found"]
    ]
    .copy()
)


print(
    "\nMissing images:",
    len(missing_images)
)


if len(missing_images) > 0:

    print(
        "\nMissing by dataset:"
    )

    print(
        missing_images[
            "dataset"
        ].value_counts()
    )

    print(
        "\nFirst missing records:"
    )

    print(
        missing_images.head(
            20
        ).to_string(
            index=False
        )
    )

else:

    print(
        "\n✓ Every caption image has a corresponding file"
    )

MISSING IMAGE ANALYSIS

Missing images: 0

✓ Every caption image has a corresponding file


8 — Create Existing Image Dataset

In [ ]:
# ============================================================
# KEEP ONLY IMAGES THAT ACTUALLY EXIST
# ============================================================

valid_path_images_df = (
    images_df[
        images_df["file_found"]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print("=" * 70)
print("VALID FILE PATH DATASET")
print("=" * 70)

print(
    "\nImages with existing files:",
    len(valid_path_images_df)
)

print(
    "Images excluded:",
    len(images_df)
    - len(valid_path_images_df)
)


print("\nDataset distribution:")

print(
    valid_path_images_df[
        "dataset"
    ].value_counts()
)


print("\n✓ Existing image paths prepared")

VALID FILE PATH DATASET

Images with existing files: 39874
Images excluded: 0

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: count, dtype: int64

✓ Existing image paths prepared


9 — Validate Image Readability

In [ ]:
# ============================================================
# IMAGE READABILITY VALIDATION
# ============================================================

print("=" * 70)
print("VALIDATING IMAGE FILES")
print("=" * 70)


def validate_image(image_path):

    try:

        with Image.open(
            image_path
        ) as image:

            image.verify()

        return True

    except Exception:

        return False


validity_results = []


for image_path in tqdm(
    valid_path_images_df[
        "image_path"
    ],
    desc="Validating images"
):

    validity_results.append(
        validate_image(
            image_path
        )
    )


valid_path_images_df[
    "image_valid"
] = validity_results


print(
    "\nTotal checked:",
    len(valid_path_images_df)
)

print(
    "Valid images:",
    valid_path_images_df[
        "image_valid"
    ].sum()
)

print(
    "Invalid/corrupted images:",
    (~valid_path_images_df[
        "image_valid"
    ]).sum()
)


print("\n✓ Image validation complete")

VALIDATING IMAGE FILES


Validating images:   0%|          | 0/39874 [00:00<?, ?it/s]


Total checked: 39874
Valid images: 39874
Invalid/corrupted images: 0

✓ Image validation complete


10 — Inspect Invalid Images

In [ ]:
# ============================================================
# INVALID IMAGE REPORT
# ============================================================

invalid_images = (
    valid_path_images_df[
        ~valid_path_images_df[
            "image_valid"
        ]
    ]
    .copy()
)


print("=" * 70)
print("INVALID IMAGE REPORT")
print("=" * 70)

print(
    "\nInvalid images:",
    len(invalid_images)
)


if len(invalid_images) > 0:

    print(
        "\nInvalid image records:\n"
    )

    print(
        invalid_images.to_string(
            index=False
        )
    )

else:

    print(
        "\n✓ No corrupted or unreadable images found"
    )

INVALID IMAGE REPORT

Invalid images: 0

✓ No corrupted or unreadable images found


11 — Create Final Verified Image Manifest

In [ ]:
# ============================================================
# CREATE FINAL VERIFIED IMAGE MANIFEST
# ============================================================

print("=" * 70)
print("CREATING VERIFIED IMAGE MANIFEST")
print("=" * 70)


verified_images_df = (
    valid_path_images_df[
        valid_path_images_df[
            "image_valid"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# CLEAN OUTPUT COLUMNS
# ------------------------------------------------------------

verified_images_df = (
    verified_images_df[
        [
            "image_id",
            "dataset",
            "image_name",
            "image_path"
        ]
    ]
)


print(
    "\nVerified images:",
    len(verified_images_df)
)

print(
    "Unique image IDs:",
    verified_images_df[
        "image_id"
    ].nunique()
)


print("\nDataset distribution:")

print(
    verified_images_df[
        "dataset"
    ].value_counts()
)


assert (
    verified_images_df[
        "image_path"
    ].notna()
    .all()
)

assert (
    verified_images_df[
        "image_id"
    ].nunique()
    == len(verified_images_df)
)


print("\n✓ Verified image manifest created")

CREATING VERIFIED IMAGE MANIFEST

Verified images: 39874
Unique image IDs: 39874

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: count, dtype: int64

✓ Verified image manifest created


12 — Save Image Manifest

In [ ]:
# ============================================================
# SAVE IMAGE MANIFEST
# ============================================================

IMAGES_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


IMAGE_MANIFEST_PATH = (
    IMAGES_OUTPUT_DIR
    / "image_manifest.csv"
)


verified_images_df.to_csv(
    IMAGE_MANIFEST_PATH,
    index=False
)


print("=" * 70)
print("IMAGE MANIFEST SAVED")
print("=" * 70)

print("\nPath:")
print(IMAGE_MANIFEST_PATH)

print(
    "\nShape:",
    verified_images_df.shape
)

print(
    "\nFile size:",
    f"{IMAGE_MANIFEST_PATH.stat().st_size / (1024 * 1024):.2f} MB"
)


print("\n✓ Image manifest saved")

IMAGE MANIFEST SAVED

Path:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Images/image_manifest.csv

Shape: (39874, 4)

File size: 5.91 MB

✓ Image manifest saved


13 — Final Notebook 4 Verification

In [ ]:
# ============================================================
# NOTEBOOK 4 — FINAL VERIFICATION
# ============================================================

final_images_df = pd.read_csv(
    IMAGE_MANIFEST_PATH
)


print("=" * 70)
print("NOTEBOOK 4 — FINAL VERIFICATION")
print("=" * 70)


print(
    "\nShape:",
    final_images_df.shape
)


print(
    "\nColumns:"
)

print(
    list(
        final_images_df.columns
    )
)


print(
    "\nTotal verified images:"
)

print(
    len(final_images_df)
)


print(
    "\nUnique image IDs:"
)

print(
    final_images_df[
        "image_id"
    ].nunique()
)


print(
    "\nDataset distribution:"
)

print(
    final_images_df[
        "dataset"
    ].value_counts()
)


print(
    "\nMissing image paths:"
)

print(
    final_images_df[
        "image_path"
    ].isna()
    .sum()
)


# ------------------------------------------------------------
# VERIFY SAMPLE FILES STILL EXIST
# ------------------------------------------------------------

sample_paths = (
    final_images_df[
        "image_path"
    ]
    .sample(
        min(
            10,
            len(final_images_df)
        ),
        random_state=42
    )
)


missing_sample_paths = 0


for image_path in sample_paths:

    if not Path(
        image_path
    ).exists():

        missing_sample_paths += 1


print(
    "\nMissing files in verification sample:",
    missing_sample_paths
)


assert (
    final_images_df[
        "image_id"
    ].nunique()
    == len(final_images_df)
)

assert (
    final_images_df[
        "image_path"
    ].notna()
    .all()
)

assert (
    missing_sample_paths == 0
)


print("\n" + "=" * 70)
print("✓ NOTEBOOK 4 COMPLETE")
print("=" * 70)

print("\nOutput:")
print(IMAGE_MANIFEST_PATH)

NOTEBOOK 4 — FINAL VERIFICATION

Shape: (39874, 4)

Columns:
['image_id', 'dataset', 'image_name', 'image_path']

Total verified images:
39874

Unique image IDs:
39874

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: count, dtype: int64

Missing image paths:
0

Missing files in verification sample: 0

✓ NOTEBOOK 4 COMPLETE

Output:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Images/image_manifest.csv
